# RAY-IMAGE N5 — Standalone Training Session (gated cross-attention)

**One session = one experiment.** This notebook runs N5 only.

N5 = a per-block learnable scalar gate on the DiT token-level text cross-attention
residual (`x = x + gate * cross`), gate init = 1.0 so the model starts identical to N2.

It warm-starts from the **existing N2 checkpoint** restored from Google Drive and trains
the SAME 8000-step budget as N2. It does **not** retrain the VAE, N2, N3, or N4.

Select a GPU runtime before starting.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. In Colab choose Runtime → Change runtime type → GPU.')
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
%cd /content
!rm -rf anime-ai-companion
!git clone -q https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!git fetch -q origin arena/01a07cdc-anime-ai-companion
!git checkout -q arena/01a07cdc-anime-ai-companion
!git rev-parse --short HEAD
!pip install -q -r requirements.txt
print('Active branch ready.')


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/RAY_IMAGE')
DRIVE_CKPT = DRIVE_ROOT / 'checkpoints'
DRIVE_N5 = DRIVE_ROOT / 'runs/N5'
DRIVE_N5.mkdir(parents=True, exist_ok=True)

# Restore the persisted N2 baseline (never retrain it).
src = DRIVE_CKPT / 'ray_image_v0_2_whiten.pt'
dst = Path('/content/ray_image_v0_2_whiten.pt')
if not src.exists():
    raise FileNotFoundError(f'Persistent N2 checkpoint not found: {src}')
shutil.copy2(src, dst)
print('Restored N2 baseline:', dst)
print('N5 Drive root:', DRIVE_ROOT)


In [ ]:
# Prepare the deterministic toy dataset (same manifest N2 used). Rebuilding the
# dataset is cheap/deterministic and is NOT model retraining.
!rm -rf data/toy
!python tools/make_toy_dataset.py --output data/toy --samples 2048 --size 64 --seed 1337
print('Toy dataset ready: data/toy/manifest.jsonl')


In [ ]:
# N5 compatibility/load smoke test (random-init models only; proves the N2->gated
# load path, gate init=1.0, forward/shape/mask, and checkpoint round-trip).
!python -m ray_image.n5_smoke
print('N5 compatibility smoke test passed.')


In [ ]:
# N5 — train ONLY N5, warm-started from the N2 baseline (8000 steps, same budget as N2).
!python -m ray_image.train_n5 \
  --manifest data/toy/manifest.jsonl \
  --checkpoint /content/ray_image_v0_2_whiten.pt \
  --steps 8000 \
  --batch-size 16 \
  --save /content/ray_image_v0_3_n5.pt \
  --seed 0
print('N5 training complete: /content/ray_image_v0_3_n5.pt')


In [ ]:
# Save the N5 checkpoint persistently (separate from the N2 baseline).
from pathlib import Path
import shutil
n5_local = Path('/content/ray_image_v0_3_n5.pt')
if not n5_local.exists():
    raise FileNotFoundError(n5_local)
n5_drive = Path('/content/drive/MyDrive/RAY_IMAGE/checkpoints/ray_image_v0_3_n5.pt')
shutil.copy2(n5_local, n5_drive)
print('Persistent N5 checkpoint:', n5_drive)


In [ ]:
# Generate the fixed 12-prompt suite from the N5 checkpoint using the SAME
# settings/seed as N2 (steps=50, seed=42) for a controlled comparison.
from pathlib import Path
import subprocess, sys
prompts = [
    ('red_circle', 'a red circle'), ('red_square', 'a red square'), ('red_triangle', 'a red triangle'),
    ('green_circle', 'a green circle'), ('green_square', 'a green square'), ('green_triangle', 'a green triangle'),
    ('blue_circle', 'a blue circle'), ('blue_square', 'a blue square'), ('blue_triangle', 'a blue triangle'),
    ('yellow_circle', 'a yellow circle'), ('yellow_square', 'a yellow square'), ('yellow_triangle', 'a yellow triangle'),
]
out_dir = Path('/content/ray_suite_n5')
out_dir.mkdir(parents=True, exist_ok=True)
for name, prompt in prompts:
    out = out_dir / f'{name}.png'
    cmd = [sys.executable, '-m', 'ray_image.generate', '--checkpoint', '/content/ray_image_v0_3_n5.pt', '--prompt', prompt, '--steps', '50', '--seed', '42', '--output', str(out)]
    subprocess.run(cmd, check=True)
print('Generated', len(list(out_dir.glob('*.png'))), 'images.')


In [ ]:
# Evaluate the fixed 12-class suite on the N5 outputs.
!python tools/evaluate_toy_suite.py --dir /content/ray_suite_n5


In [ ]:
# Persist the N5 evaluator output and an image grid to Drive runs/N5.
from pathlib import Path
import subprocess, sys
import shutil

DRIVE_N5 = Path('/content/drive/MyDrive/RAY_IMAGE/runs/N5')
result = subprocess.run([sys.executable, 'tools/evaluate_toy_suite.py', '--dir', '/content/ray_suite_n5'],
                        capture_output=True, text=True, check=True)
print(result.stdout)
eval_path = DRIVE_N5 / 'evaluator_output.txt'
eval_path.write_text(result.stdout)

from PIL import Image, ImageDraw
files = sorted(Path('/content/ray_suite_n5').glob('*.png'))
cols = 4
rows = (len(files) + cols - 1) // cols
sheet = Image.new('RGB', (cols * 192, rows * 220), 'white')
draw = ImageDraw.Draw(sheet)
for i, p in enumerate(files):
    im = Image.open(p).convert('RGB').resize((192, 192))
    x = (i % cols) * 192; y = (i // cols) * 220
    sheet.paste(im, (x, y)); draw.text((x + 5, y + 196), p.stem, fill='black')
grid_path = DRIVE_N5 / 'N5_image_grid.png'
sheet.save(grid_path)
print('Persistent evaluator output:', eval_path)
print('Persistent image grid:', grid_path)


## Done

Send back the printed evaluator metrics (color / shape / suite accuracy), the
`N5_image_grid.png`, and the N5 checkpoint path. No training metrics are reported
from the implementation stage -- these results come only from this real Colab run.